# FlightGraph — OpenFlights Dataset Exploration

## 1. Introduction

FlightGraph is an airline network resilience and disruption simulation
project built using Neo4j and Python.

Before designing the graph database, we need to understand the
structure, quality, and limitations of the OpenFlights dataset.

### Research objectives

This notebook investigates:

1. Dataset size and structure.
2. Airport identifier uniqueness and missing values.
3. Route integrity and missing airport references.
4. Duplicate routes and airline information.
5. Implications for Neo4j graph modeling.

### Data sources

- `airports.dat`: Airport information.
- `routes.dat`: Airline route information.

Source: https://github.com/jpatokal/openflights

**Limitation:** OpenFlights contains historical route information
and should not be interpreted as a current flight schedule.

In [1]:
from pathlib import Path

import pandas as pd

DATA_DIR = Path("data/raw")

print("Working directory:", Path.cwd())
print("Data directory exists:", DATA_DIR.exists())

Working directory: /Users/camilletanguy/Projects/flightgraph
Data directory exists: True


## 2. Airport Dataset Exploration

We first examine the structure and quality of the airport dataset.

Our objectives are to:
- Identify the number of airport records.
- Verify the uniqueness of airport identifiers.
- Measure missing values.
- Determine which attributes are suitable for Neo4j.

In [2]:
airport_columns = [
    "airport_id",
    "name",
    "city",
    "country",
    "iata",
    "icao",
    "latitude",
    "longitude",
    "altitude",
    "timezone",
    "dst",
    "tz_database",
    "type",
    "source",
]

airports = pd.read_csv(
    DATA_DIR / "airports.dat",
    header=None,
    names=airport_columns,
    na_values=[r"\N"],
)

airports.head()

,airport_id,name,city,country,iata,icao,latitude,longitude,altitude,timezone,dst,tz_database,type,source
0,1,Goroka Airport,Goroka,Papua New Guinea,GKA,AYGA,-6.081690,145.391998,5282,10.0,U,Pacific/Port_Moresby,airport,OurAirports
1,2,Madang Airport,Madang,Papua New Guinea,MAG,AYMD,-5.207080,145.789001,20,10.0,U,Pacific/Port_Moresby,airport,OurAirports
2,3,Mount Hagen Kagamuga Airport,Mount Hagen,Papua New Guinea,HGU,AYMH,-5.826790,144.296005,5388,10.0,U,Pacific/Port_Moresby,airport,OurAirports
3,4,Nadzab Airport,Nadzab,Papua New Guinea,LAE,AYNZ,-6.569803,146.725977,239,10.0,U,Pacific/Port_Moresby,airport,OurAirports
4,5,Port Moresby Jacksons International Airport,Port Moresby,Papua New Guinea,POM,AYPY,-9.443380,147.220001,146,10.0,U,Pacific/Port_Moresby,airport,OurAirports


In [3]:
print("Total airports:", len(airports))
print("Unique airport IDs:", airports["airport_id"].nunique())
print("Duplicate airport IDs:", airports["airport_id"].duplicated().sum())
print("Missing airport IDs:", airports["airport_id"].isna().sum())
print("Missing IATA codes:", airports["iata"].isna().sum())

Total airports: 7698
Unique airport IDs: 7698
Duplicate airport IDs: 0
Missing airport IDs: 0
Missing IATA codes: 1626


### Findings

The dataset contains 7,698 airports with 7,698 unique airport IDs
and no missing airport IDs.

However, 1,626 airports have missing IATA codes.

**Modeling implication:** OpenFlights airport IDs are a better
candidate for identifying Airport nodes than IATA codes.

We will validate route references before finalizing this decision.

## 3. Route Dataset Exploration

The routes dataset describes airline connections between airports.

We will investigate:
- The number of route records.
- Missing source and destination airport IDs.
- Whether routes reference airports present in our airport dataset.
- Duplicate connections between airports.

These checks will help us design reliable relationships in Neo4j.

In [4]:
route_columns = [
    "airline",
    "airline_id",
    "source_airport",
    "source_airport_id",
    "destination_airport",
    "destination_airport_id",
    "codeshare",
    "stops",
    "equipment",
]

routes = pd.read_csv(
    DATA_DIR / "routes.dat",
    header=None,
    names=route_columns,
    na_values=[r"\N"],
)

routes.head()

,airline,airline_id,source_airport,source_airport_id,destination_airport,destination_airport_id,codeshare,stops,equipment
0,2B,410.0,AER,2965.0,KZN,2990.0,NaN,0,CR2
1,2B,410.0,ASF,2966.0,KZN,2990.0,NaN,0,CR2
2,2B,410.0,ASF,2966.0,MRV,2962.0,NaN,0,CR2
3,2B,410.0,CEK,2968.0,KZN,2990.0,NaN,0,CR2
4,2B,410.0,CEK,2968.0,OVB,4078.0,NaN,0,CR2


In [5]:
print("Total routes:", len(routes))

print(
    "Missing source airport IDs:",
    routes["source_airport_id"].isna().sum()
)

print(
    "Missing destination airport IDs:",
    routes["destination_airport_id"].isna().sum()
)

Total routes: 67663
Missing source airport IDs: 220
Missing destination airport IDs: 221


### 3.1 Route Reference Integrity

Before importing routes into Neo4j, we need to verify that their source
and destination airport IDs correspond to airports in `airports.dat`.

A route may contain an airport ID that is not present in the airport
dataset, even when the ID itself is not missing.

We therefore check whether both airport IDs exist in our airport dataset.

**Objective:** Determine how many routes can be reliably represented
as relationships between Airport nodes.

In [6]:
airport_ids = set(airports["airport_id"])

valid_sources = routes["source_airport_id"].isin(airport_ids)
valid_destinations = routes["destination_airport_id"].isin(airport_ids)

valid_routes = valid_sources & valid_destinations

print("Routes with valid airport references:", valid_routes.sum())
print("Routes with invalid airport references:", (~valid_routes).sum())

Routes with valid airport references: 66771
Routes with invalid airport references: 892


### Findings — Route Reference Integrity

The dataset contains 67,663 route records.

Of these, 66,771 routes (98.68%) reference airports that exist
in `airports.dat`, while 892 routes (1.32%) have at least one
invalid airport reference.

This suggests that most routes can be represented as
airport-to-airport relationships in Neo4j.

However, invalid references may result from either:
- Missing airport IDs.
- Airport IDs that are absent from the airport dataset.

Further investigation is required before defining our
data-cleaning strategy.

### 3.2 Missing vs. Unknown Airport References

An invalid airport reference can occur because an ID is missing
or because an ID does not exist in the airport dataset.

We distinguish these situations to determine whether routes
should be excluded or whether their identifiers might be
recoverable.

In [7]:
missing_ids = (
    routes["source_airport_id"].isna()
    | routes["destination_airport_id"].isna()
)

unknown_ids = (
    (
        routes["source_airport_id"].notna()
        & ~valid_sources
    )
    | (
        routes["destination_airport_id"].notna()
        & ~valid_destinations
    )
)

print("Routes with at least one missing airport ID:", missing_ids.sum())
print("Routes with at least one unknown airport ID:", unknown_ids.sum())
print("Routes with both issues:", (missing_ids & unknown_ids).sum())

Routes with at least one missing airport ID: 423
Routes with at least one unknown airport ID: 476
Routes with both issues: 7


#### Findings — Missing and Unknown Airport References

Of the 67,663 route records:

- 423 have at least one missing airport ID.
- 476 reference at least one airport ID absent from `airports.dat`.
- 7 exhibit both issues.

Using the inclusion-exclusion principle, 892 routes have
at least one invalid airport reference.

**Provisional cleaning decision:**

For the initial graph, retain the 66,771 routes whose source
and destination airport IDs both exist in the airport dataset.

This avoids introducing artificial connections. The 892 excluded
records will be documented as a limitation of the dataset.

### 3.3 Duplicate Airport Connections

Multiple airlines may operate routes between the same pair
of airports.

We investigate how many distinct directed airport connections
exist after filtering invalid airport references.

This will help us determine how to represent routes and
airlines in Neo4j.

In [8]:
clean_routes = routes[valid_routes].copy()

unique_connections = clean_routes.drop_duplicates(
    subset=["source_airport_id", "destination_airport_id"]
)

print("Valid route records:", len(clean_routes))
print("Unique directed airport connections:", len(unique_connections))
print(
    "Additional records sharing an airport connection:",
    len(clean_routes) - len(unique_connections)
)

Valid route records: 66771
Unique directed airport connections: 36907
Additional records sharing an airport connection: 29864


#### Findings — Duplicate Airport Connections

After filtering invalid airport references, the dataset contains:

- 66,771 valid route records.
- 36,907 unique directed airport connections.
- 29,864 additional route records sharing an airport connection.

Approximately 44.7% of valid route records share their
source–destination pair with another record.

**Modeling implication:**

Airport connectivity analysis can use one directed relationship
per unique airport pair.

However, individual route records may contain additional
information, particularly airline identifiers.

The final graph model must balance analytical simplicity
with information preservation.

### 3.4 Airline Diversity on Airport Connections

We investigate whether multiple route records connecting the same
airports represent different airlines or duplicate records from
the same airline.

This analysis will help determine whether airline information
should be represented explicitly in our graph.

In [9]:
unique_airline_routes = clean_routes.drop_duplicates(
    subset=[
        "source_airport_id",
        "destination_airport_id",
        "airline",
    ]
)

print("Valid route records:", len(clean_routes))

print(
    "Unique airport-airline connections:",
    len(unique_airline_routes)
)

print(
    "Repeated records for the same airline and airport pair:",
    len(clean_routes) - len(unique_airline_routes)
)

print(
    "Missing airline IDs:",
    clean_routes["airline_id"].isna().sum()
)

Valid route records: 66771
Unique airport-airline connections: 66771
Repeated records for the same airline and airport pair: 0
Missing airline IDs: 455


#### Findings — Airline Diversity

All 66,771 valid route records have unique combinations of
source airport, destination airport, and airline code.

This indicates that repeated airport-to-airport connections
represent different airline codes rather than duplicate
records under this definition.

Additionally, 455 valid routes have missing numerical airline IDs.

**Modeling implication:**

A graph containing only unique airport-to-airport connections
would lose airline-specific information unless it were
preserved separately.

We should investigate airline identifiers before finalizing
the Neo4j schema.

### 3.5 Airline Identifier Completeness

Some route records have missing numerical airline IDs.

We investigate whether these records still contain airline
codes, which could provide an alternative way to identify
the operating airline.

In [10]:
missing_airline_id = clean_routes["airline_id"].isna()

print(
    "Routes with missing airline IDs:",
    missing_airline_id.sum()
)

print(
    "Routes missing both airline ID and code:",
    (
        missing_airline_id
        & clean_routes["airline"].isna()
    ).sum()
)

clean_routes.loc[
    missing_airline_id,
    ["airline", "airline_id", "source_airport", "destination_airport"]
].head(10)

Routes with missing airline IDs: 455
Routes missing both airline ID and code: 0


,airline,airline_id,source_airport,destination_airport
312,3H,NaN,AKV,YIK
313,3H,NaN,AKV,YPX
314,3H,NaN,XGR,YVP
315,3H,NaN,YGL,YGW
316,3H,NaN,YGL,YPX
317,3H,NaN,YGL,YUL
318,3H,NaN,YGW,YGL
319,3H,NaN,YGW,YPH
320,3H,NaN,YGW,YSK
321,3H,NaN,YGW,YUD


#### Findings — Airline Identifier Completeness

Among the 66,771 valid route records:

- 455 have missing numerical airline IDs.
- None of these 455 records have a missing airline code.

Therefore, all routes with missing airline IDs retain some
airline information.

**Modeling implication:**

Airline codes can help preserve information when numerical
airline IDs are unavailable.

However, we must verify the consistency and uniqueness of
airline codes before using them as identifiers for Airline nodes.

### 3.6 Airline Code Consistency

Before selecting a unique identifier for Airline nodes,
we investigate whether individual airline codes correspond
to multiple numerical airline IDs.

This helps determine whether airline codes can be used
safely as identifiers in our Neo4j graph.

In [11]:
airline_code_mapping = (
    clean_routes
    .groupby("airline")["airline_id"]
    .nunique()
)

ambiguous_codes = airline_code_mapping[
    airline_code_mapping > 1
]

print("Unique airline codes:", clean_routes["airline"].nunique())
print("Codes linked to multiple airline IDs:", len(ambiguous_codes))

ambiguous_codes.head(10)

Unique airline codes: 566
Codes linked to multiple airline IDs: 0


Series([], Name: airline_id, dtype: int64)

#### Findings — Airline Code Consistency

The valid routes dataset contains 566 unique airline codes.

No airline code is associated with multiple non-missing
numerical airline IDs.

This suggests that airline codes are suitable candidates
for identifying Airline nodes.

However, the reverse mapping (airline ID to airline code)
has not yet been verified.

In [12]:
airline_id_mapping = (
    clean_routes
    .dropna(subset=["airline_id"])
    .groupby("airline_id")["airline"]
    .nunique()
)

ambiguous_airline_ids = airline_id_mapping[
    airline_id_mapping > 1
]

print("Unique known airline IDs:", len(airline_id_mapping))
print(
    "Airline IDs linked to multiple codes:",
    len(ambiguous_airline_ids)
)

ambiguous_airline_ids.head(10)

Unique known airline IDs: 546
Airline IDs linked to multiple codes: 0


Series([], Name: airline, dtype: int64)

#### Findings — Airline Identifier Mapping

The dataset contains 566 unique airline codes and
546 unique non-missing airline IDs.

No airline code maps to multiple known IDs, and no known
airline ID maps to multiple codes.

**Modeling implication:**

Airline codes are suitable candidate identifiers for
Airline nodes in this dataset.

Numerical airline IDs can be retained as optional properties.

## 4. Proposed Neo4j Graph Model

### 4.1 Graph Structure

FlightGraph will initially use three node labels:

| Node label | Description | Identifier |
|---|---|---|
| Airport | Represents an airport | airport_id |
| Airline | Represents an airline | airline code |
| Country | Represents a country | country name |

The graph will contain three relationship types:

| Relationship | Source | Target | Description |
|---|---|---|---|
| FLIES_TO | Airport | Airport | Represents a directed airport connection |
| LOCATED_IN | Airport | Country | Identifies the country containing an airport |
| OPERATES | Airline | Airport | Indicates that an airline serves an airport |

### 4.2 Modeling Rationale

Our dataset exploration identified:

- 7,698 airports with unique, non-missing airport IDs.
- 66,771 routes with valid airport references.
- 36,907 unique directed airport connections.
- 566 distinct airline codes.

**Airport-to-airport connections**

We propose creating one directed FLIES_TO relationship per unique
airport pair.

When several airlines operate the same connection, their codes
will be stored as a list property on the relationship.

This simplifies pathfinding, centrality analysis, and airport
closure simulations while preserving information about the
airlines serving each connection.

**Airline representation**

Airline nodes will use airline codes as provisional identifiers.

OPERATES relationships will link each airline to the airports
it serves. These relationships do not independently represent
individual airline routes.

**Country representation**

Country nodes will support geographic exploration and
country-level disruption analysis.

### 4.3 Limitations

- OpenFlights contains historical connections, not live schedules.
- Aggregated FLIES_TO relationships do not represent individual flights.
- Airline codes are assumed to be consistent within this dataset.
- Route filtering excludes records without valid airport references.
- The proposed model must be validated before implementation.

**Status: PROVISIONAL — not yet implemented or validated.**

### 4.4 Country Data Quality

Our proposed graph model includes Country nodes connected
to Airport nodes through LOCATED_IN relationships.

Before implementing this structure, we need to verify:

- Whether any airports have missing country information.
- How many distinct country values exist.
- Whether country names appear consistent.

This analysis will help us determine whether the country
column can be used to create Country nodes.

In [13]:
print("Total airports:", len(airports))

print(
    "Missing country values:",
    airports["country"].isna().sum()
)

print(
    "Unique country values:",
    airports["country"].nunique()
)

airports["country"].value_counts().head(10)

Total airports: 7698
Missing country values: 0
Unique country values: 237


country
United States     1512
Canada             430
Australia          334
Brazil             264
Russia             264
Germany            249
China              241
France             217
United Kingdom     167
India              148
Name: count, dtype: int64

#### Findings — Country Data Quality

The dataset contains 237 distinct country values across 7,698 airports.

No airport records have missing country information.

The United States has the most airport records (1,512),
followed by Canada (430) and Australia (334).

**Modeling implication:**

Country nodes can be created from the distinct country values,
and each Airport node can be linked to its country through
a LOCATED_IN relationship.

Country names will initially be preserved as provided by
OpenFlights. These values should not be interpreted as
standardized country identifiers.

### 4.5 Airline–Airport Relationships

The proposed OPERATES relationship indicates that an airline
serves an airport appearing as either the source or destination
of one of its recorded routes.

We count distinct airline–airport pairs to estimate the
number of OPERATES relationships required.

In [14]:
source_pairs = clean_routes[
    ["airline", "source_airport_id"]
].rename(columns={"source_airport_id": "airport_id"})

destination_pairs = clean_routes[
    ["airline", "destination_airport_id"]
].rename(columns={"destination_airport_id": "airport_id"})

airline_airport_pairs = pd.concat(
    [source_pairs, destination_pairs],
    ignore_index=True,
).drop_duplicates()

print(
    "Unique airline-airport relationships:",
    len(airline_airport_pairs)
)

Unique airline-airport relationships: 19146


#### Findings — Airline–Airport Relationships

The dataset contains 19,146 unique airline–airport pairs.

An OPERATES relationship will connect an Airline node to
each airport appearing as the source or destination of
one of its recorded routes.

This relationship indicates that the airline serves the
airport; it does not imply that the airport is an airline hub.

### Expected Graph Size

| Element | Expected count |
|---|---:|
| Airport nodes | 7,698 |
| Airline nodes | 566 |
| Country nodes | 237 |
| FLIES_TO relationships | 36,907 |
| LOCATED_IN relationships | 7,698 |
| OPERATES relationships | 19,146 |

These counts are estimates derived from the source data.
They must be verified after Neo4j ingestion.

### 4.6 Comparison of Graph Modeling Alternatives

Two main approaches were considered for representing airline routes.

| Criterion | Route Nodes | Direct Airport Relationships |
|---|---|---|
| Airport connectivity queries | More complex | Simple |
| Shortest-path analysis | Requires additional modeling | Direct |
| Airport closure simulation | More complex | Straightforward |
| Airline-specific information | Preserved individually | Aggregated into relationship properties |
| Graph size | Larger | Smaller |
| Implementation complexity | Higher | Lower |

### Selected Architecture

I selected a graph with three node labels:

- `Airport`
- `Airline`
- `Country`

And three relationship types:

- `FLIES_TO`: Airport → Airport
- `LOCATED_IN`: Airport → Country
- `OPERATES`: Airline → Airport

Each `FLIES_TO` relationship represents a unique directed
airport connection and stores the associated airline codes.

### Decision Rationale

FlightGraph primarily investigates airport connectivity,
network resilience, and the consequences of airport closures.

Direct airport-to-airport relationships simplify pathfinding
and graph algorithms while avoiding unnecessary complexity.

Airline and Country nodes provide additional analytical
capabilities without complicating the core airport network.

### Trade-offs and Limitations

This architecture prioritizes airport-level analysis over
individual airline-route modeling.

Although airline codes are preserved, individual route
records are not represented as separate graph entities.

The resulting network represents historical connectivity,
not flight frequency, passenger capacity, or current schedules.

**Decision status: PROVISIONAL**

The architecture will be validated through Neo4j ingestion
and representative graph queries.

## 5. Machine Learning Feasibility

### 5.1 Airport Network Sparsity

We are considering link prediction as the machine learning
component of FlightGraph.

The objective would be to classify whether a directed airport
connection exists in the historical network.

Before selecting an ML approach, we investigate network sparsity.

A missing connection is not necessarily a genuinely negative
example, and random train/test splitting can introduce
data leakage.

The results of this analysis will inform our ML methodology.

In [15]:
connected_airports = set(clean_routes["source_airport_id"]) | set(
    clean_routes["destination_airport_id"]
)

n_airports = len(connected_airports)
n_connections = len(unique_connections)

possible_connections = n_airports * (n_airports - 1)
density = n_connections / possible_connections

print("Airports participating in routes:", n_airports)
print("Unique directed connections:", n_connections)
print("Possible directed connections:", possible_connections)
print(f"Network density: {density:.4%}")

Airports participating in routes: 3214
Unique directed connections: 36907
Possible directed connections: 10326582
Network density: 0.3574%


In [16]:
self_loops = (
    unique_connections["source_airport_id"]
    == unique_connections["destination_airport_id"]
)

print("Self-loop connections:", self_loops.sum())

Self-loop connections: 1


### 5.2 Self-loop Investigation

The airport network contains one directed connection
whose source and destination airport IDs are identical.

We inspect this record to determine whether it represents
a meaningful route or a data-quality anomaly.

Self-loops will be excluded from the initial connectivity
graph because they do not contribute to reachability
between distinct airports.

In [17]:
self_loop_routes = clean_routes[
    clean_routes["source_airport_id"]
    == clean_routes["destination_airport_id"]
]

self_loop_routes

,airline,airline_id,source_airport,source_airport_id,destination_airport,destination_airport_id,codeshare,stops,equipment
33276,IL,10121.0,PKN,3910.0,PKN,3910.0,NaN,0,AT7


#### Findings — Self-loop Investigation

One self-loop was identified in the valid routes dataset:

- Airline: IL
- Source airport: PKN (ID 3910)
- Destination airport: PKN (ID 3910)
- Aircraft equipment: AT7

The source and destination airport IDs are identical.

**Cleaning decision:**

Exclude this self-loop from the airport connectivity graph
because it does not contribute to reachability between
distinct airports.

The record will remain in the original dataset for traceability.

After excluding the self-loop, the network contains
36,906 unique directed connections between distinct airports.

The corrected network density is approximately 0.3574%.

In [18]:
network_routes = clean_routes[
    clean_routes["source_airport_id"]
    != clean_routes["destination_airport_id"]
].copy()

network_connections = network_routes.drop_duplicates(
    subset=["source_airport_id", "destination_airport_id"]
)

print("Route records after filtering:", len(network_routes))
print("Unique directed connections:", len(network_connections))

Route records after filtering: 66770
Unique directed connections: 36906


### 5.3 Machine Learning Feasibility Assessment

#### Proposed Task

FlightGraph will investigate binary link prediction on the
historical airport network.

The objective is to predict whether a directed connection
exists between two airports.

The model will be evaluated on held-out connections rather
than presented as a predictor of future airline routes.

#### Dataset Characteristics

- 3,214 airports participate in valid routes.
- 36,906 unique directed connections remain after cleaning.
- Network density is approximately 0.3574%.
- The graph is highly sparse.

#### Methodological Challenges

**Negative sampling:** Airport pairs without recorded routes
are unobserved connections, not necessarily genuine negatives.

**Data leakage:** Network features must be computed using
training connections only. Held-out connections must not
influence feature construction.

**Evaluation:** Accuracy alone is unsuitable because the
network is highly sparse. We will consider precision,
recall, F1-score, and PR-AUC.

**Temporal limitations:** OpenFlights does not provide reliable
route-opening dates for a temporal train/test split.

#### Provisional Methodology

1. Split observed connections into training and test sets.
2. Sample unobserved airport pairs as candidate negatives.
3. Construct features from airport attributes and the
   training graph only.
4. Train a simple classification model.
5. Compare its performance against a baseline.
6. Evaluate and document limitations.

**Status: PROVISIONAL**

Link prediction appears feasible as a historical
connection-reconstruction experiment, but the sampling,
feature construction, and evaluation methodology must be
validated before implementation.